# MotorPH 3rd Quarter Sales Data for 2025

## Loading Data Set

In [ ]:
import pandas as pd

sales_data = pd.read_csv("raw_data/MotorPH_Sales Data-3rd Quarter-Year 2025.csv")

print(sales_data.head().to_string())

## Data Set Information

### Data Summary

In [ ]:
print("Sales Data Information")
print(sales_data.info())
print()
print("Sales Data Summary")
print(sales_data.describe())
print()
print("Missing Values per column")
print(pd.isnull(sales_data).sum())

## Data Preprocessing

Rename the columns

In [ ]:
old_date = "date"
old_product_name = "product"
old_unit_price = "unitprice"
old_product_quantity = "quantity"
old_total_price = "total"
old_payment = "payment"
old_client_type = "client_type"

new_date = "Date"
new_product_name = "Product Name"
new_unit_price = "Unit Price"
new_product_quantity = "Product Quantity"
new_total_price = "Total Price"
new_payment = "Payment Type"
new_client_type = "Client Type"

sales_data.rename(columns={
    old_date : new_date,
    old_product_name : new_product_name,
    old_unit_price : new_unit_price,
    old_product_quantity : new_product_quantity,
    old_total_price : new_total_price,
    old_payment : new_payment,
    old_client_type : new_client_type
}, inplace=True)

print(sales_data.head().to_string())

### Check for malformed data

In [ ]:
print("Checking malformed data in client type column")
print(sorted(sales_data[new_client_type].dropna().unique()))
print()
print("Checking malformed data in product name column")
for product in sorted(sales_data[new_product_name].unique()):
    print(product)

### Find the malformed product name
Find and clean the malformed data based on close match. Based on the previous code most malformed data ends with **x** replacing the last digit of the product name.

In [ ]:
products = sales_data[new_product_name].unique()

malformed = [p for p in products if p.endswith("x")]
clean = [p for p in products if not p.endswith("x")]

product_fixes = {}

for m in malformed:
    prefix = m[:-1]  # everything except the last character
    matches = [c for c in clean if c.startswith(prefix)]
    print(f"{m!r:30} -> possible match: {matches}")
    print("fixing...")
    if len(matches) == 1:
        product_fixes[m] = matches[0]
    else:
        print(f"Ambiguous or no match for {m!r}: {matches}")

print(product_fixes)

### Replaced the malformed data using the potential match

In [ ]:
sales_data[new_product_name] = sales_data[new_product_name].replace(product_fixes)
remaining = [p for p in sales_data[new_product_name].unique() if p.endswith("x")]
print("Remaining malformed products:")
print(remaining)

#### Null value handling
To handle the missing values in client_type, instead of dropping the affected rows, I identified the specific product each client purchased and compared it against other clients who purchased the same product. The client type (Wholesale or Retail) was then inferred based on which type had the higher number of transactions for that product, supported by the average quantity purchased per client type.

In [ ]:
missing_client_type = sales_data[sales_data[new_client_type].isnull()]
print(missing_client_type[new_product_name])

In [ ]:
missing_products = missing_client_type[new_product_name].tolist()

related_sales = sales_data[
    (sales_data[new_product_name].isin(missing_products)) &
    (sales_data[new_client_type].notna())
]

print(related_sales[[new_product_name, new_client_type]].to_string())

Missing value of the Client Type will be based on the average quantity purchased by retails and wholesale on a specific product.

In [ ]:
filtered_sales = sales_data[sales_data[new_product_name].isin(missing_products)]

summary = filtered_sales.groupby([new_product_name, new_client_type]).agg(
    Transaction_Count=(new_product_quantity, "count"),
    Total_Quantity=(new_product_quantity, "sum"),
    Average_Quantity=(new_product_quantity, "mean")
).unstack()

print(summary.to_string())

Fill the missing value of Client type based on average quantity purchased by client type.

In [ ]:
def infer_client_type(row):
    product = row[new_product_name]
    quantity = row[new_product_quantity]

    retail_avg = avg_quantity.loc[product, "Retail"]
    wholesale_avg = avg_quantity.loc[product, "Wholesale"]

    if abs(quantity - retail_avg) <= abs(quantity - wholesale_avg):
        return "Retail"
    else:
        return "Wholesale"

In [ ]:
avg_quantity = summary["Average_Quantity"]
print(avg_quantity)

In [ ]:
mask = sales_data[new_client_type].isnull()

sales_data.loc[mask, new_client_type] = sales_data.loc[mask].apply(infer_client_type, axis=1)

print(sales_data[new_client_type].isnull().sum())  # should be 0
print(sales_data.loc[mask.index[mask], [new_product_name, new_product_quantity, new_client_type]].to_string())

### Missing Value for payment type

Same approach will also be applied to Payment Type. Missing values will be replaced by the frequent payment type the client (retail or wholesale) used.